# 05. Integración MILP de las capas Rho y Pi

Este notebook extiende la formulación MILP de la capa `theta` incorporando
las transformaciones `rho` y `pi` de Keccak.

A diferencia de `theta`, las capas `rho` y `pi` no realizan operaciones XOR
entre varios bits. Ambas capas únicamente reorganizan las posiciones del
estado:

- `rho` rota circularmente los bits dentro de cada lane;
- `pi` permuta las coordenadas de los lanes.

Por esta razón, su formulación MILP puede expresarse mediante igualdades
directas entre variables binarias.

El flujo estudiado será:

$$
A
\longrightarrow
\theta
\longrightarrow
\rho
\longrightarrow
\pi.
$$

La implementación se validará comparando la solución obtenida por el
solver con las funciones de referencia implementadas en Python.

In [1]:
# ============================================================
# CONFIGURACIÓN DEL ENTORNO DEL NOTEBOOK
# ============================================================

from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """
    Busca hacia arriba el directorio raíz del proyecto.

    Se considera raíz el directorio que contiene la carpeta `src`.
    """
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").exists():
            return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto con una carpeta `src`."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Raíz del proyecto:", PROJECT_ROOT)
print("Directorio src:", SRC_DIR)

Raíz del proyecto: D:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada
Directorio src: D:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\src


In [22]:
# ============================================================
# REVISIÓN DE LA API IMPLEMENTADA
# ============================================================

from keccak_milp.layers import (
    theta,
    rho,
    pi,
    rho_pi,
    rho_pi_destination,
)
from keccak_milp.model import KeccakMILPModel


layer_names = [
    name
    for name in dir(layers)
    if any(term in name.lower() for term in ("theta", "rho", "pi"))
]

model_names = [
    name
    for name in dir(KeccakMILPModel)
    if any(term in name.lower() for term in ("theta", "rho", "pi"))
]

print("Funciones disponibles en keccak_milp.layers:")
for name in layer_names:
    print(" -", name)

print("\nMétodos disponibles en KeccakMILPModel:")
for name in model_names:
    print(" -", name)

Funciones disponibles en keccak_milp.layers:
 - RHO_OFFSETS
 - pi
 - pi_destination
 - rho
 - rho_offset
 - rho_pi
 - rho_pi_destination
 - theta
 - theta_effect

Métodos disponibles en KeccakMILPModel:
 - _add_rho_pi_constraints
 - _add_theta_c_constraints
 - _add_theta_d_constraints
 - _add_theta_output_constraints
 - _create_rho_pi_variables
 - _create_theta_variables
 - add_rho_pi_layers
 - add_theta_layer
 - rho_pi_output_values
 - rho_pi_output_variable
 - theta_output_values
 - theta_output_variable


In [23]:
# ============================================================
# VALIDACIÓN DE FUNCIONES NECESARIAS
# ============================================================

required_layer_functions = {
    "rho_pi",
    "rho_pi_destination",
}

available_layer_functions = set(dir(layers))
missing_functions = required_layer_functions - available_layer_functions

assert not missing_functions, (
    "Faltan funciones necesarias en keccak_milp.layers: "
    f"{sorted(missing_functions)}"
)

print("Funciones mínimas de Rho-Pi disponibles correctamente.")

Funciones mínimas de Rho-Pi disponibles correctamente.


## Formulación MILP de las capas Rho y Pi

Sea $T[x,y,k]$ el estado binario obtenido después de la capa `theta`, donde:

$$
x,y \in \{0,1,2,3,4\},
$$

y:

$$
k \in \{0,1,\ldots,z-1\}.
$$

La transformación `rho` aplica una rotación circular diferente a cada lane.
Si $r[x,y]$ es el desplazamiento correspondiente al lane $(x,y)$, entonces
el bit situado en la posición $k$ se desplaza a:

$$
k' = (k+r[x,y]) \bmod z.
$$

Posteriormente, la transformación `pi` mueve el lane ubicado en $(x,y)$ a
las coordenadas:

$$
(x',y') =
\left(
y,\,
(2x+3y)\bmod 5
\right).
$$

Por tanto, la transformación combinada puede expresarse como:

$$
B\left[
y,\,
(2x+3y)\bmod 5,\,
(k+r[x,y])\bmod z
\right]
=
T[x,y,k],
$$

donde $B$ representa el estado después de aplicar `rho` y `pi`.

Como estas dos capas solamente permutan bits, cada restricción MILP es una
igualdad entre dos variables binarias. No se necesitan variables auxiliares
para representar operaciones XOR.

In [24]:
# ============================================================
# INSPECCIÓN DE LAS FIRMAS DE LA API ACTUAL
# ============================================================

import inspect

from keccak_milp.layers import (
    rho,
    pi,
    rho_pi,
    rho_pi_destination,
)
from keccak_milp.model import KeccakMILPModel


objects_to_inspect = {
    "KeccakMILPModel": KeccakMILPModel,
    "KeccakMILPModel.add_theta_layer": KeccakMILPModel.add_theta_layer,
    "KeccakMILPModel.add_rho_pi_layers": KeccakMILPModel.add_rho_pi_layers,
    "KeccakMILPModel.theta_output_values": (
        KeccakMILPModel.theta_output_values
    ),
    "KeccakMILPModel.rho_pi_output_values": (
        KeccakMILPModel.rho_pi_output_values
    ),
    "rho": rho,
    "pi": pi,
    "rho_pi": rho_pi,
    "rho_pi_destination": rho_pi_destination,
}


for name, obj in objects_to_inspect.items():
    print(f"{name}{inspect.signature(obj)}")

KeccakMILPModel(config: 'ExperimentConfig', name: 'str | None' = None) -> 'None'
KeccakMILPModel.add_theta_layer(self, round_index: 'int') -> 'None'
KeccakMILPModel.add_rho_pi_layers(self, round_index: 'int') -> 'None'
KeccakMILPModel.theta_output_values(self, round_index: 'int', tolerance: 'float' = 0.5) -> 'list[list[list[int]]]'
KeccakMILPModel.rho_pi_output_values(self, round_index: 'int', tolerance: 'float' = 0.5) -> 'list[list[list[int]]]'
rho(state: 'NDArray[T]') -> 'NDArray[T]'
pi(state: 'NDArray[T]') -> 'NDArray[T]'
rho_pi(state: 'NDArray[T]') -> 'NDArray[T]'
rho_pi_destination(x: 'int', y: 'int', k: 'int', z: 'int') -> 'tuple[int, int, int]'


In [25]:
# ============================================================
# DOCUMENTACIÓN DE LOS MÉTODOS PRINCIPALES
# ============================================================

main_methods = [
    KeccakMILPModel.add_theta_layer,
    KeccakMILPModel.add_rho_pi_layers,
    KeccakMILPModel.rho_pi_output_values,
    rho_pi_destination,
]


for method in main_methods:
    print("=" * 70)
    print(f"Objeto: {method.__qualname__}")
    print("-" * 70)

    documentation = inspect.getdoc(method)

    if documentation:
        print(documentation)
    else:
        print("No se encontró un docstring para este objeto.")

    print()

Objeto: KeccakMILPModel.add_theta_layer
----------------------------------------------------------------------
Crea y conecta la capa theta de una ronda.

La operación es idempotente.

Objeto: KeccakMILPModel.add_rho_pi_layers
----------------------------------------------------------------------
Agrega las capas rho y pi después de theta.

Requiere que la capa theta de la misma ronda ya exista.
La operación es idempotente.

Objeto: KeccakMILPModel.rho_pi_output_values
----------------------------------------------------------------------
Recupera la salida de rho y pi como una estructura 5 × 5 × z.

Objeto: rho_pi_destination
----------------------------------------------------------------------
Calcula directamente el destino de un bit después de rho y pi.

Esta función será útil posteriormente para construir las igualdades
del modelo MILP sin necesidad de crear operaciones NumPy.

Returns
-------
tuple[int, int, int]
    Coordenadas destino después de rho y pi.



In [26]:
# ============================================================
# VERIFICACIÓN DEL MAPEO DE DESTINO RHO-PI
# ============================================================

z = 8

test_positions = [
    (0, 0, 0),
    (1, 0, 0),
    (0, 1, 0),
    (2, 3, 4),
    (4, 4, 7),
]


print(f"Longitud de lane utilizada para la prueba: z = {z}\n")

for x, y, k in test_positions:
    destination = rho_pi_destination(x, y, k, z)

    print(
        f"Origen ({x}, {y}, {k}) "
        f"-> Destino {destination}"
    )

Longitud de lane utilizada para la prueba: z = 8

Origen (0, 0, 0) -> Destino (0, 0, 0)
Origen (1, 0, 0) -> Destino (0, 2, 1)
Origen (0, 1, 0) -> Destino (1, 3, 4)
Origen (2, 3, 4) -> Destino (3, 3, 3)
Origen (4, 4, 7) -> Destino (4, 0, 5)


## Verificación de la propiedad de permutación

Las capas `rho` y `pi` no crean ni eliminan bits. Cada posición del estado
de entrada debe tener exactamente una posición de destino, y cada posición
de salida debe recibir exactamente un bit.

Para una longitud de lane $z$, el estado contiene:

$$
25z
$$

posiciones binarias.

Por tanto, la función `rho_pi_destination` debe generar:

- $25z$ posiciones de origen distintas;
- $25z$ posiciones de destino distintas;
- ninguna colisión entre destinos;
- ninguna posición de salida sin asignar.

Esta propiedad permite modelar ambas capas mediante igualdades directas.

In [27]:
# ============================================================
# PRUEBA EXHAUSTIVA DE LA PERMUTACIÓN RHO-PI
# ============================================================

z = 8

all_positions = {
    (x, y, k)
    for x in range(5)
    for y in range(5)
    for k in range(z)
}

rho_pi_mapping = {
    (x, y, k): rho_pi_destination(x, y, k, z)
    for x in range(5)
    for y in range(5)
    for k in range(z)
}

origins = set(rho_pi_mapping.keys())
destinations = set(rho_pi_mapping.values())

expected_positions = 25 * z

assert len(origins) == expected_positions
assert len(destinations) == expected_positions
assert origins == all_positions
assert destinations == all_positions

print("Prueba de biyección completada correctamente.")
print(f"Posiciones de origen:  {len(origins)}")
print(f"Posiciones de destino: {len(destinations)}")
print(f"Colisiones detectadas: {len(rho_pi_mapping) - len(destinations)}")

Prueba de biyección completada correctamente.
Posiciones de origen:  200
Posiciones de destino: 200
Colisiones detectadas: 0


In [28]:
# ============================================================
# MUESTRA DE LOS PRIMEROS MAPEOS RHO-PI
# ============================================================

print("Origen         -> Destino")
print("-" * 32)

for index, (origin, destination) in enumerate(rho_pi_mapping.items()):
    print(f"{str(origin):14} -> {destination}")

    if index == 11:
        break

Origen         -> Destino
--------------------------------
(0, 0, 0)      -> (0, 0, 0)
(0, 0, 1)      -> (0, 0, 1)
(0, 0, 2)      -> (0, 0, 2)
(0, 0, 3)      -> (0, 0, 3)
(0, 0, 4)      -> (0, 0, 4)
(0, 0, 5)      -> (0, 0, 5)
(0, 0, 6)      -> (0, 0, 6)
(0, 0, 7)      -> (0, 0, 7)
(0, 1, 0)      -> (1, 3, 4)
(0, 1, 1)      -> (1, 3, 5)
(0, 1, 2)      -> (1, 3, 6)
(0, 1, 3)      -> (1, 3, 7)


## Tamaño esperado de la capa MILP

Sea $T[x,y,k]$ la salida de `theta`. Para representar la salida de
`rho` y `pi`, se crea una variable binaria:

$$
B[x,y,k] \in \{0,1\}
$$

por cada posición del estado.

Como existen $25z$ posiciones, la capa debe agregar:

$$
N_{\mathrm{variables}}^{\rho\pi}=25z.
$$

Además, cada variable de salida se conecta con exactamente una variable
de entrada mediante una igualdad:

$$
B[x',y',k']=T[x,y,k].
$$

Por tanto, la cantidad de restricciones añadidas debe ser:

$$
N_{\mathrm{restricciones}}^{\rho\pi}=25z.
$$

Para $z=8$, se esperan 200 variables binarias nuevas y 200 restricciones
de igualdad nuevas.

In [29]:
# ============================================================
# FUNCIONES AUXILIARES PARA INSPECCIONAR EL MODELO
# ============================================================

from keccak_milp.config import ExperimentConfig


def count_problem_variables(model: KeccakMILPModel) -> int:
    """Cuenta las variables actualmente conectadas al problema."""
    return len(model.problem.variables())


def count_problem_constraints(model: KeccakMILPModel) -> int:
    """
    Cuenta las restricciones de forma compatible con distintas
    versiones de PuLP.
    """
    constraints_object = model.problem.constraints

    if callable(constraints_object):
        return len(constraints_object())

    return len(constraints_object)


def add_layer_for_round(method, round_index: int) -> None:
    """
    Ejecuta un método de capa para una ronda.

    La llamada posicional evita depender del nombre exacto usado
    para el parámetro de ronda.
    """
    method(round_index)

In [30]:
# ============================================================
# MODELO DE REFERENCIA: SOLAMENTE HASTA THETA
# ============================================================

z = 8
round_index = 0

config_theta = ExperimentConfig(
    z=z,
    rounds=1,
    solver="cbc",
    verbose=False,
)

theta_model = KeccakMILPModel(config_theta)

add_layer_for_round(
    theta_model.add_theta_layer,
    round_index,
)

theta_variable_count = count_problem_variables(theta_model)
theta_constraint_count = count_problem_constraints(theta_model)

print("Modelo construido hasta Theta")
print("-" * 40)
print(f"Variables:     {theta_variable_count}")
print(f"Restricciones: {theta_constraint_count}")

Modelo construido hasta Theta
----------------------------------------
Variables:     760
Restricciones: 280


In [31]:
# ============================================================
# MODELO COMPLETO: THETA SEGUIDO DE RHO-PI
# ============================================================

config_rho_pi = ExperimentConfig(
    z=z,
    rounds=1,
    solver="cbc",
    verbose=False,
)

rho_pi_model = KeccakMILPModel(config_rho_pi)

add_layer_for_round(
    rho_pi_model.add_theta_layer,
    round_index,
)

add_layer_for_round(
    rho_pi_model.add_rho_pi_layers,
    round_index,
)

rho_pi_variable_count = count_problem_variables(rho_pi_model)
rho_pi_constraint_count = count_problem_constraints(rho_pi_model)

print("Modelo construido hasta Rho-Pi")
print("-" * 40)
print(f"Variables:     {rho_pi_variable_count}")
print(f"Restricciones: {rho_pi_constraint_count}")

Modelo construido hasta Rho-Pi
----------------------------------------
Variables:     960
Restricciones: 480


In [32]:
# ============================================================
# VALIDACIÓN DEL TAMAÑO DE LA CAPA RHO-PI
# ============================================================

added_variables = rho_pi_variable_count - theta_variable_count
added_constraints = rho_pi_constraint_count - theta_constraint_count

expected_rho_pi_size = 25 * z

print("Incremento producido por Rho-Pi")
print("-" * 40)
print(f"Variables añadidas:     {added_variables}")
print(f"Restricciones añadidas: {added_constraints}")
print(f"Valor esperado:         {expected_rho_pi_size}")

assert added_variables == expected_rho_pi_size, (
    "La capa Rho-Pi no agregó la cantidad esperada de variables: "
    f"esperadas={expected_rho_pi_size}, obtenidas={added_variables}."
)

assert added_constraints == expected_rho_pi_size, (
    "La capa Rho-Pi no agregó la cantidad esperada de restricciones: "
    f"esperadas={expected_rho_pi_size}, obtenidas={added_constraints}."
)

print("\nLa formulación Rho-Pi tiene el tamaño esperado.")

Incremento producido por Rho-Pi
----------------------------------------
Variables añadidas:     200
Restricciones añadidas: 200
Valor esperado:         200

La formulación Rho-Pi tiene el tamaño esperado.


In [33]:
# ============================================================
# VALIDACIÓN DE IDEMPOTENCIA
# ============================================================

variables_before = count_problem_variables(rho_pi_model)
constraints_before = count_problem_constraints(rho_pi_model)

add_layer_for_round(
    rho_pi_model.add_rho_pi_layers,
    round_index,
)

variables_after = count_problem_variables(rho_pi_model)
constraints_after = count_problem_constraints(rho_pi_model)

assert variables_after == variables_before
assert constraints_after == constraints_before

print("Idempotencia verificada correctamente.")
print(f"Variables antes y después:     {variables_before}")
print(f"Restricciones antes y después: {constraints_before}")

Idempotencia verificada correctamente.
Variables antes y después:     960
Restricciones antes y después: 480


## Validación funcional con una entrada controlada

La validación estructural comprobó que las capas `rho` y `pi` agregan la
cantidad esperada de variables y restricciones. Sin embargo, también es
necesario verificar que esas restricciones representen correctamente la
transformación de Keccak.

Para ello se seguirá el siguiente procedimiento:

1. construir un estado binario de entrada conocido;
2. fijar todos sus bits en el modelo MILP;
3. agregar las capas `theta`, `rho` y `pi`;
4. resolver el problema con CBC;
5. recuperar las salidas del modelo;
6. compararlas con las funciones de referencia en Python.

La comparación se realizará bit a bit:

$$
T_{\mathrm{MILP}}[x,y,k]
=
T_{\mathrm{ref}}[x,y,k],
$$

y:

$$
B_{\mathrm{MILP}}[x,y,k]
=
B_{\mathrm{ref}}[x,y,k].
$$

Además, como `rho` y `pi` solamente permutan posiciones, debe cumplirse:

$$
\operatorname{HW}(B)
=
\operatorname{HW}(T),
$$

donde $\operatorname{HW}$ representa el peso de Hamming.

In [40]:
# ============================================================
# FUNCIONES AUXILIARES PARA MANIPULAR ESTADOS
# ============================================================

from copy import deepcopy
import inspect

from pulp import LpStatus, lpSum


def create_zero_state(z: int) -> np.ndarray:
    """
    Crea un estado binario Keccak con forma 5 × 5 × z.
    """
    if z <= 0:
        raise ValueError("La longitud de lane z debe ser positiva.")

    return np.zeros(
        (5, 5, z),
        dtype=np.int64,
    )


def normalize_binary_state(
    state: list[list[list[float | int]]],
    z: int,
) -> list[list[list[int]]]:
    """
    Convierte los valores de una solución MILP a enteros binarios.
    """
    normalized = create_zero_state(z)

    for x in range(5):
        for y in range(5):
            for k in range(z):
                value = state[x][y][k]

                if value is None:
                    raise ValueError(
                        f"El bit ({x}, {y}, {k}) no tiene valor."
                    )

                normalized[x][y][k] = int(round(value))

    return normalized


def hamming_weight(
    state: list[list[list[int]]],
    z: int,
) -> int:
    """Calcula el número de bits activos del estado."""
    return sum(
        state[x][y][k]
        for x in range(5)
        for y in range(5)
        for k in range(z)
    )


def differing_positions(
    first_state: list[list[list[int]]],
    second_state: list[list[list[int]]],
    z: int,
) -> list[tuple[int, int, int, int, int]]:
    """
    Devuelve las posiciones en las que dos estados son diferentes.

    Cada elemento contiene:
        (x, y, k, valor_primero, valor_segundo)
    """
    differences = []

    for x in range(5):
        for y in range(5):
            for k in range(z):
                first_value = first_state[x][y][k]
                second_value = second_state[x][y][k]

                if first_value != second_value:
                    differences.append(
                        (
                            x,
                            y,
                            k,
                            first_value,
                            second_value,
                        )
                    )

    return differences


print("Funciones auxiliares definidas correctamente.")

Funciones auxiliares definidas correctamente.


In [35]:
# ============================================================
# FUNCIONES DE COMPATIBILIDAD CON LA API ACTUAL
# ============================================================

def get_state_variable(
    model: KeccakMILPModel,
    round_index: int,
    x: int,
    y: int,
    k: int,
):
    """
    Recupera una variable del estado de frontera.

    Se utiliza primero el método público `state_variable`.
    """
    state_method = getattr(model, "state_variable", None)

    if callable(state_method):
        return state_method(round_index, x, y, k)

    key = (round_index, x, y, k)

    if key in model.state:
        return model.state[key]

    raise AttributeError(
        "No fue posible recuperar la variable del estado "
        f"para la posición {key}."
    )


def call_reference_layer(
    layer_function,
    state: list[list[list[int]]],
    z: int,
):
    """
    Ejecuta una capa de referencia considerando si recibe o no `z`.
    """
    signature = inspect.signature(layer_function)
    parameter_count = len(signature.parameters)

    if parameter_count >= 2:
        return layer_function(state, z)

    return layer_function(state)


def get_round_output(output_method, round_index: int):
    """
    Recupera una salida intermedia considerando si el método
    recibe explícitamente el índice de ronda.
    """
    signature = inspect.signature(output_method)

    if len(signature.parameters) == 0:
        return output_method()

    return output_method(round_index)


print("API detectada:")
print(
    "state_variable:",
    inspect.signature(KeccakMILPModel.state_variable),
)
print(
    "solve:",
    inspect.signature(KeccakMILPModel.solve),
)
print(
    "theta_output_values:",
    inspect.signature(
        KeccakMILPModel.theta_output_values
    ),
)
print(
    "rho_pi_output_values:",
    inspect.signature(
        KeccakMILPModel.rho_pi_output_values
    ),
)

API detectada:
state_variable: (self, round_index: 'int', x: 'int', y: 'int', k: 'int') -> 'pulp.LpVariable'
solve: (self) -> 'str'
theta_output_values: (self, round_index: 'int', tolerance: 'float' = 0.5) -> 'list[list[list[int]]]'
rho_pi_output_values: (self, round_index: 'int', tolerance: 'float' = 0.5) -> 'list[list[list[int]]]'


In [36]:
# ============================================================
# CONSTRUCCIÓN DE UNA ENTRADA BINARIA CONTROLADA
# ============================================================

z = 8
round_index = 0

input_state = create_zero_state(z)

active_input_bits = [
    (0, 0, 0),
    (1, 0, 0),
    (0, 1, 3),
    (2, 3, 4),
    (3, 2, 5),
    (4, 4, 7),
]

for x, y, k in active_input_bits:
    input_state[x][y][k] = 1


print("Bits activos de la entrada:")

for position in active_input_bits:
    print(" -", position)

print(
    "\nPeso de Hamming de la entrada:",
    hamming_weight(input_state, z),
)

Bits activos de la entrada:
 - (0, 0, 0)
 - (1, 0, 0)
 - (0, 1, 3)
 - (2, 3, 4)
 - (3, 2, 5)
 - (4, 4, 7)

Peso de Hamming de la entrada: 6


In [41]:
# ============================================================
# RESULTADOS ESPERADOS MEDIANTE LAS FUNCIONES DE REFERENCIA
# ============================================================

import numpy as np

from keccak_milp.layers import theta, rho_pi


# Convertir el estado construido con listas a un arreglo NumPy
input_state_np = np.asarray(
    input_state,
    dtype=np.int64,
)

assert input_state_np.shape == (5, 5, z), (
    "La entrada debe tener forma (5, 5, z). "
    f"Forma recibida: {input_state_np.shape}"
)


# ------------------------------------------------------------
# Aplicar las capas de referencia
# ------------------------------------------------------------

theta_reference = theta(
    input_state_np.copy(),
)

rho_pi_reference = rho_pi(
    theta_reference.copy(),
)


# ------------------------------------------------------------
# Normalizar las salidas a valores enteros binarios
# ------------------------------------------------------------

theta_reference = normalize_binary_state(
    theta_reference,
    z,
)

rho_pi_reference = normalize_binary_state(
    rho_pi_reference,
    z,
)


# ------------------------------------------------------------
# Mostrar resultados
# ------------------------------------------------------------

theta_reference_weight = hamming_weight(
    theta_reference,
    z,
)

rho_pi_reference_weight = hamming_weight(
    rho_pi_reference,
    z,
)

print("Resultados de referencia")
print("-" * 45)
print(
    "Peso después de Theta:  ",
    theta_reference_weight,
)
print(
    "Peso después de Rho-Pi: ",
    rho_pi_reference_weight,
)


# Rho y Pi únicamente permutan bits
assert theta_reference_weight == rho_pi_reference_weight, (
    "Rho-Pi debería conservar el peso de Hamming "
    "de la salida de Theta."
)

print(
    "\nRho-Pi conserva correctamente "
    "el peso de Hamming."
)

Resultados de referencia
---------------------------------------------
Peso después de Theta:   64
Peso después de Rho-Pi:  64

Rho-Pi conserva correctamente el peso de Hamming.


In [46]:
# ============================================================
# CONSTRUCCIÓN DEL MODELO DE VALIDACIÓN
# ============================================================

import numpy as np

validation_config = ExperimentConfig(
    z=z,
    rounds=1,
    solver="cbc",
    verbose=False,
)

# Convertir la entrada a NumPy, independientemente de cómo fue creada
input_state_np = np.asarray(
    input_state,
    dtype=np.int64,
)

assert input_state_np.shape == (5, 5, z), (
    "El estado inicial debe tener forma (5, 5, z). "
    f"Forma recibida: {input_state_np.shape}."
)

assert np.all(np.isin(input_state_np, [0, 1])), (
    "El estado inicial debe contener únicamente valores binarios."
)


# Crear el modelo desde cero
validation_model = KeccakMILPModel(validation_config)

# Construir el esqueleto para registrar el objetivo correctamente
validation_model.build_skeleton()

# Estas operaciones son idempotentes
validation_model.add_theta_layer(round_index)
validation_model.add_rho_pi_layers(round_index)


# ------------------------------------------------------------
# Fijar todos los bits del estado inicial
# ------------------------------------------------------------

fixed_input_bits = 0

for x in range(5):
    for y in range(5):
        for k in range(z):
            input_variable = get_state_variable(
                validation_model,
                round_index,
                x,
                y,
                k,
            )

            input_value = int(input_state_np[x, y, k])

            validation_model.problem += (
                input_variable == input_value,
                f"fix_input_r{round_index}_{x}_{y}_{k}",
            )

            fixed_input_bits += 1


print("Modelo de validación construido.")
print("-" * 45)
print(
    "Variables:",
    count_problem_variables(validation_model),
)
print(
    "Restricciones:",
    count_problem_constraints(validation_model),
)
print(
    "Bits de entrada fijados:",
    fixed_input_bits,
)

assert fixed_input_bits == 25 * z

Modelo de validación construido.
---------------------------------------------
Variables: 960
Restricciones: 681
Bits de entrada fijados: 200


In [47]:
# ============================================================
# RESOLUCIÓN DEL MODELO CON CBC
# ============================================================

solve_result = validation_model.solve()

status_code = validation_model.problem.status
status_name = LpStatus.get(
    status_code,
    str(status_code),
)

print("Resultado devuelto por solve():", solve_result)
print("Código de estado:", status_code)
print("Estado del solver:", status_name)

assert status_name == "Optimal", (
    "Se esperaba una solución óptima, "
    f"pero CBC devolvió: {status_name}."
)

Resultado devuelto por solve(): Optimal
Código de estado: 1
Estado del solver: Optimal


In [48]:
# ============================================================
# RECUPERACIÓN DE LAS SALIDAS DEL MODELO MILP
# ============================================================

theta_milp = get_round_output(
    validation_model.theta_output_values,
    round_index,
)

rho_pi_milp = get_round_output(
    validation_model.rho_pi_output_values,
    round_index,
)

theta_milp = normalize_binary_state(
    theta_milp,
    z,
)

rho_pi_milp = normalize_binary_state(
    rho_pi_milp,
    z,
)


print("Pesos de Hamming obtenidos por el MILP")
print("-" * 45)
print(
    "Salida de Theta:  ",
    hamming_weight(theta_milp, z),
)
print(
    "Salida de Rho-Pi: ",
    hamming_weight(rho_pi_milp, z),
)

Pesos de Hamming obtenidos por el MILP
---------------------------------------------
Salida de Theta:   64
Salida de Rho-Pi:  64


In [49]:
# ============================================================
# COMPARACIÓN MILP FRENTE A LA IMPLEMENTACIÓN DE REFERENCIA
# ============================================================

theta_differences = differing_positions(
    theta_milp,
    theta_reference,
    z,
)

rho_pi_differences = differing_positions(
    rho_pi_milp,
    rho_pi_reference,
    z,
)


print("Comparación bit a bit")
print("-" * 45)
print(
    "Diferencias en Theta:  ",
    len(theta_differences),
)
print(
    "Diferencias en Rho-Pi: ",
    len(rho_pi_differences),
)


if theta_differences:
    print("\nPrimeras diferencias de Theta:")

    for difference in theta_differences[:10]:
        print(difference)


if rho_pi_differences:
    print("\nPrimeras diferencias de Rho-Pi:")

    for difference in rho_pi_differences[:10]:
        print(difference)


assert not theta_differences, (
    "La formulación MILP de Theta no coincide "
    "con la implementación de referencia."
)

assert not rho_pi_differences, (
    "La formulación MILP de Rho-Pi no coincide "
    "con la implementación de referencia."
)

assert (
    hamming_weight(theta_milp, z)
    ==
    hamming_weight(rho_pi_milp, z)
)

print(
    "\nValidación funcional completada correctamente."
)

Comparación bit a bit
---------------------------------------------
Diferencias en Theta:   0
Diferencias en Rho-Pi:  0

Validación funcional completada correctamente.


## Validación sobre múltiples entradas

La prueba anterior utilizó un único estado binario controlado. Aunque este
resultado confirma la equivalencia para ese caso, es conveniente repetir la
comparación sobre diferentes estados de entrada.

Se generarán estados binarios pseudoaleatorios utilizando una semilla fija.
Esto permite que el experimento sea completamente reproducible.

Para cada estado se comprobará que:

$$
T_{\mathrm{MILP}} = T_{\mathrm{ref}},
$$

$$
B_{\mathrm{MILP}} = B_{\mathrm{ref}},
$$

y:

$$
\operatorname{HW}(T_{\mathrm{MILP}})
=
\operatorname{HW}(B_{\mathrm{MILP}}).
$$

La igualdad se evaluará sobre las $25z$ posiciones del estado.

In [50]:
# ============================================================
# FUNCIÓN REUTILIZABLE PARA RESOLVER THETA + RHO-PI
# ============================================================

import numpy as np

from pulp import LpStatus


def solve_theta_rho_pi_case(
    input_state: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, str]:
    """
    Resuelve mediante MILP las capas Theta, Rho y Pi para un
    estado binario fijo.

    Parameters
    ----------
    input_state:
        Estado NumPy binario con forma (5, 5, z).

    Returns
    -------
    tuple
        Salida de Theta, salida de Rho-Pi y estado del solver.
    """
    input_array = np.asarray(
        input_state,
        dtype=np.int64,
    )

    if input_array.ndim != 3:
        raise ValueError(
            "El estado debe tener tres dimensiones."
        )

    if input_array.shape[:2] != (5, 5):
        raise ValueError(
            "Las dos primeras dimensiones deben ser 5 × 5."
        )

    if not np.all(np.isin(input_array, [0, 1])):
        raise ValueError(
            "El estado debe contener únicamente valores binarios."
        )

    local_z = input_array.shape[2]
    local_round = 0

    local_config = ExperimentConfig(
        z=local_z,
        rounds=1,
        solver="cbc",
        verbose=False,
    )

    local_model = KeccakMILPModel(local_config)

    # Registra la estructura y la función objetivo del modelo.
    local_model.build_skeleton()

    # Las operaciones son idempotentes.
    local_model.add_theta_layer(local_round)
    local_model.add_rho_pi_layers(local_round)

    # Fijar completamente la entrada.
    for x in range(5):
        for y in range(5):
            for k in range(local_z):
                variable = get_state_variable(
                    local_model,
                    local_round,
                    x,
                    y,
                    k,
                )

                value = int(input_array[x, y, k])

                local_model.problem += (
                    variable == value,
                    f"fix_case_input_{x}_{y}_{k}",
                )

    solve_result = local_model.solve()

    status_code = local_model.problem.status
    status_name = LpStatus.get(
        status_code,
        str(status_code),
    )

    if status_name != "Optimal":
        raise RuntimeError(
            "CBC no encontró una solución óptima. "
            f"Estado obtenido: {status_name}."
        )

    theta_output = np.asarray(
        get_round_output(
            local_model.theta_output_values,
            local_round,
        ),
        dtype=np.int64,
    )

    rho_pi_output = np.asarray(
        get_round_output(
            local_model.rho_pi_output_values,
            local_round,
        ),
        dtype=np.int64,
    )

    theta_output = np.rint(theta_output).astype(
        np.int64
    )

    rho_pi_output = np.rint(rho_pi_output).astype(
        np.int64
    )

    return (
        theta_output,
        rho_pi_output,
        str(solve_result),
    )


print("Función de resolución definida correctamente.")

Función de resolución definida correctamente.


In [51]:
# ============================================================
# PRUEBA DE LA FUNCIÓN REUTILIZABLE
# ============================================================

theta_function_output, rho_pi_function_output, function_status = (
    solve_theta_rho_pi_case(input_state_np)
)

assert np.array_equal(
    theta_function_output,
    np.asarray(theta_reference),
)

assert np.array_equal(
    rho_pi_function_output,
    np.asarray(rho_pi_reference),
)

print("Estado del solver:", function_status)
print(
    "Peso de Theta:",
    int(theta_function_output.sum()),
)
print(
    "Peso de Rho-Pi:",
    int(rho_pi_function_output.sum()),
)
print(
    "La función reutilizable reproduce el caso controlado."
)

Estado del solver: Optimal
Peso de Theta: 64
Peso de Rho-Pi: 64
La función reutilizable reproduce el caso controlado.


In [52]:
# ============================================================
# VALIDACIÓN SOBRE ESTADOS PSEUDOALEATORIOS
# ============================================================

random_seed = 2026
number_of_cases = 5
z = 8

rng = np.random.default_rng(random_seed)

validation_results = []


for case_index in range(number_of_cases):
    random_input = rng.integers(
        low=0,
        high=2,
        size=(5, 5, z),
        dtype=np.int64,
    )

    # Implementación de referencia
    theta_expected = theta(
        random_input.copy()
    )

    rho_pi_expected = rho_pi(
        theta_expected.copy()
    )

    # Implementación MILP
    theta_obtained, rho_pi_obtained, solver_status = (
        solve_theta_rho_pi_case(random_input)
    )

    theta_equal = np.array_equal(
        theta_obtained,
        theta_expected,
    )

    rho_pi_equal = np.array_equal(
        rho_pi_obtained,
        rho_pi_expected,
    )

    theta_weight = int(theta_obtained.sum())
    rho_pi_weight = int(rho_pi_obtained.sum())

    weight_preserved = (
        theta_weight == rho_pi_weight
    )

    validation_results.append(
        {
            "caso": case_index + 1,
            "estado_solver": solver_status,
            "peso_entrada": int(random_input.sum()),
            "peso_theta": theta_weight,
            "peso_rho_pi": rho_pi_weight,
            "theta_correcto": theta_equal,
            "rho_pi_correcto": rho_pi_equal,
            "peso_conservado": weight_preserved,
        }
    )

    assert theta_equal, (
        "La salida MILP de Theta no coincide "
        f"en el caso {case_index + 1}."
    )

    assert rho_pi_equal, (
        "La salida MILP de Rho-Pi no coincide "
        f"en el caso {case_index + 1}."
    )

    assert weight_preserved, (
        "Rho-Pi no conservó el peso de Hamming "
        f"en el caso {case_index + 1}."
    )


print(
    f"Se validaron correctamente "
    f"{number_of_cases} estados pseudoaleatorios."
)

Se validaron correctamente 5 estados pseudoaleatorios.


In [53]:
# ============================================================
# RESUMEN DE LA VALIDACIÓN ALEATORIA
# ============================================================

import pandas as pd


validation_dataframe = pd.DataFrame(
    validation_results
)

validation_dataframe

,caso,estado_solver,peso_entrada,peso_theta,peso_rho_pi,theta_correcto,rho_pi_correcto,peso_conservado
0,1,Optimal,97,93,93,True,True,True
1,2,Optimal,95,105,105,True,True,True
2,3,Optimal,95,97,97,True,True,True
3,4,Optimal,100,104,104,True,True,True
4,5,Optimal,95,107,107,True,True,True


In [54]:
# ============================================================
# COMPROBACIÓN GLOBAL DE LOS RESULTADOS
# ============================================================

assert validation_dataframe[
    "theta_correcto"
].all()

assert validation_dataframe[
    "rho_pi_correcto"
].all()

assert validation_dataframe[
    "peso_conservado"
].all()

assert (
    validation_dataframe["estado_solver"]
    == "Optimal"
).all()

print("Todas las validaciones fueron superadas.")
print(
    "Casos evaluados:",
    len(validation_dataframe),
)
print(
    "Errores de Theta:",
    int(
        (~validation_dataframe["theta_correcto"]).sum()
    ),
)
print(
    "Errores de Rho-Pi:",
    int(
        (~validation_dataframe["rho_pi_correcto"]).sum()
    ),
)
print(
    "Violaciones de conservación del peso:",
    int(
        (~validation_dataframe["peso_conservado"]).sum()
    ),
)

Todas las validaciones fueron superadas.
Casos evaluados: 5
Errores de Theta: 0
Errores de Rho-Pi: 0
Violaciones de conservación del peso: 0


## Conclusiones

La integración MILP de las capas `rho` y `pi` fue validada tanto
estructural como funcionalmente.

Los principales resultados son:

1. La transformación combinada `rho_pi` define una biyección sobre las
   $25z$ posiciones del estado.

2. Para $z=8$, la capa agrega exactamente:

   $$
   25z = 200
   $$

   variables binarias de salida y 200 restricciones de igualdad.

3. El método `add_rho_pi_layers` es idempotente. Una segunda llamada no
   duplica variables ni restricciones.

4. Las salidas MILP de `theta` y `rho_pi` coinciden bit a bit con las
   implementaciones de referencia.

5. Las capas `rho` y `pi` conservan el peso de Hamming de la salida de
   `theta`, ya que únicamente reorganizan las posiciones del estado.

6. La equivalencia también fue comprobada sobre varios estados binarios
   pseudoaleatorios generados con una semilla fija.

Con este resultado, el modelo representa correctamente el flujo:

$$
A
\longrightarrow
\theta
\longrightarrow
\rho
\longrightarrow
\pi.
$$

La siguiente etapa consistirá en estudiar la capa no lineal `chi`, cuya
formulación requiere representar productos o relaciones lógicas entre
variables binarias.